# Duck Local Runner — test the mods on YOUR machine
**SJSU CMPE 295A Group 2 · July 2026**

Run the exact Duck harness + our DUCK MODS experiments on the **25 public ARC-AGI-3 games**, on your own
computer — so you can A/B experiment presets for a few dollars (or free, locally) instead of burning the
one-per-day Kaggle submission. Same engine, same experiment names as `duck-mods-v2-experiments.ipynb`.

**Three ways to supply the LLM:**

| Backend | What you need | Notes |
|---|---|---|
| `openrouter` | an [OpenRouter](https://openrouter.ai) API key | easiest; pay per token; Qwen3.6-27B ≈ a few $ per full pass |
| `huggingface` | a HF token ([Inference Providers](https://huggingface.co/docs/inference-providers)) | OpenAI-compatible router at `router.huggingface.co/v1` |
| `local` | LM Studio / Ollama / vLLM / llama.cpp serving an OpenAI-compatible endpoint | free; 27B needs ≈ 18–20 GB VRAM at 4-bit; prefer a **vision** model |

**Requirements:** Python **3.12** kernel (`uv venv --python 3.12` or conda), git, and the competition folder
`arc-prize-2026-arc-agi-3/` (already in this repo) containing `environment_files/` + `arc_agi_3_wheels/`.
Works on Windows natively in API mode (no vLLM install needed). GPU is only required for the `local` backend.

**Honest caveats:** local runs use a *vision-capable* model to match Kaggle behavior (`multimodal=False` works
for text-only models but expect lower scores — Tufa found multimodality a main driver). Absolute scores here
are not comparable to the Kaggle LB (different budgets/hardware) — compare experiments **against each other**,
same backend, same settings, ≥5 passes.

In [ ]:
# ==============================================================================
#  SETTINGS - edit this cell only
# ==============================================================================

# ---- 1) Choose your LLM backend ----------------------------------------------
#   "openrouter"  : hosted API, pay per token, zero install (recommended start)
#   "huggingface" : HF Inference Providers router, pay/free tier with HF_TOKEN
#   "local"       : any OpenAI-compatible local server (LM Studio, Ollama,
#                   vLLM, llama.cpp server) - free, needs a decent GPU
BACKEND = "openrouter"

BACKENDS = {
    "openrouter": {
        "base_url": "https://openrouter.ai/api/v1",
        "api_key_env": "OPENROUTER_API_KEY",   # export it, or you'll be prompted below
        "model": "qwen/qwen3.6-27b",           # verify the exact slug at openrouter.ai/models
        "provider": "openrouter",              # OpenAI-safe payload subset
        "multimodal": True,                    # Qwen3.6 is a VLM - keep True
    },
    "huggingface": {
        "base_url": "https://router.huggingface.co/v1",
        "api_key_env": "HF_TOKEN",
        # HF router accepts "org/model" (auto-routes) or "org/model:provider".
        # Check availability at huggingface.co/models -> Inference Providers.
        "model": "Qwen/Qwen3.6-27B",
        "provider": "openrouter",              # safe payload subset works here too
        "multimodal": True,
    },
    "local": {
        # LM Studio default:  http://127.0.0.1:1234/v1
        # vLLM default:       http://127.0.0.1:8000/v1
        # Ollama default:     http://127.0.0.1:11434/v1
        "base_url": "http://127.0.0.1:1234/v1",
        "api_key_env": "",                     # most local servers need no key
        "model": "qwen3.6-27b",                # must equal the name your server serves
        "provider": "vllm",                    # full payload (top_k etc.); if your server
                                               # rejects extra fields, switch to "openrouter"
        "multimodal": True,                    # set False for text-only local models
    },
}

# ---- 2) What to run ------------------------------------------------------------
EXPERIMENT = "mods_v1"      # any preset from the experiment cell below
GAMES = "all"               # "all" = the 25 public games, or prefixes: ["ls20", "ft09", "vc33"]
N_PASSES = 1                # independent tries per game (>=5 for a real A/B verdict)
MINUTES_PER_GAME = 20       # per-game budget per pass (Kaggle uses 132)
CONCURRENT_GAMES = 4        # parallel games; APIs: 2-6 (mind rate limits), local: 1-4
TOTAL_HOURS_CAP = None      # e.g. 3.0 = hard stop after 3 h; None = no cap

# ---- 3) Paths -------------------------------------------------------------------
from pathlib import Path
REPO_DIR   = Path("duck-harness")                 # Tufa repo (auto-cloned next cell)
COMP_DIR   = Path("arc-prize-2026-arc-agi-3")     # competition folder (games + wheels)
OUTPUT_ROOT = Path("local-runs")                  # results land in a timestamped subdir

# ---- quick sanity summary --------------------------------------------------------
B = BACKENDS[BACKEND]
print(f"backend={BACKEND}  model={B['model']}  multimodal={B['multimodal']}")
print(f"experiment={EXPERIMENT}  games={GAMES}  passes={N_PASSES}  "
      f"{MINUTES_PER_GAME} min/game  x{CONCURRENT_GAMES} parallel")
_est = (25 if GAMES == "all" else len(GAMES)) * N_PASSES * MINUTES_PER_GAME / max(1, CONCURRENT_GAMES) / 60
print(f"rough wall-clock estimate: ~{_est:.1f} h (plus model latency variance)")


## 1 · Fetch / locate the harness repo and the game files

In [ ]:
# ==============================================================================
#  Fetch / locate the two ingredients: Tufa's harness repo + the game files
# ==============================================================================
import subprocess, sys

if not REPO_DIR.exists():
    print("cloning Tufalabs/duck-harness ...")
    subprocess.check_call(["git", "clone", "--depth", "1",
                           "https://github.com/Tufalabs/duck-harness.git", str(REPO_DIR)])
else:
    print(f"found harness repo: {REPO_DIR.resolve()}")

TAAF_SRC = REPO_DIR / "tufa-arc-agi-framework" / "src"
INF_SRC  = REPO_DIR / "ARC3-Inference"
ENV_DIR  = COMP_DIR / "environment_files"
WHEELS   = COMP_DIR / "arc_agi_3_wheels"

problems = []
if not TAAF_SRC.is_dir():  problems.append(f"missing {TAAF_SRC} - clone failed?")
if not INF_SRC.is_dir():   problems.append(f"missing {INF_SRC}")
if not ENV_DIR.is_dir():   problems.append(
    f"missing {ENV_DIR} - copy the competition folder here, or download it from the "
    f"Kaggle competition's Data tab (it contains environment_files/ and arc_agi_3_wheels/)")
if not WHEELS.is_dir():    problems.append(f"missing {WHEELS}")
if problems:
    raise SystemExit("SETUP PROBLEMS:\n  - " + "\n  - ".join(problems))

n_games = len([p for p in ENV_DIR.iterdir() if p.is_dir()])
print(f"OK: harness + {n_games} public game folders + wheelhouse present")
if sys.version_info[:2] != (3, 12):
    print(f"WARNING: Python {sys.version.split()[0]} detected - the harness targets 3.12. "
          f"3.10/3.11 may hit syntax/dependency errors; 3.12 strongly recommended "
          f"(e.g. `uv venv --python 3.12` or conda).")


## 2 · Install dependencies into this kernel

In [ ]:
# ==============================================================================
#  Install dependencies into THIS kernel's environment (idempotent)
# ==============================================================================
#  * arc-agi + arcengine come from the competition's own wheelhouse (pure-python)
#  * the rest are ordinary PyPI packages
#  * nothing GPU-related is needed in API mode (vLLM is NOT installed)
import subprocess, sys, shutil

def _has_pip() -> bool:
    return subprocess.run([sys.executable, "-m", "pip", "--version"],
                          capture_output=True).returncode == 0

if not _has_pip():
    # uv-created venvs ship without pip - bootstrap it, or fall back to uv itself
    if subprocess.run([sys.executable, "-m", "ensurepip", "--upgrade"],
                      capture_output=True).returncode == 0 and _has_pip():
        print("bootstrapped pip via ensurepip")
    elif shutil.which("uv"):
        print("no pip in this venv - using `uv pip` instead")
    else:
        raise SystemExit("This kernel has neither pip nor uv. Create the env with "
                         "`python -m venv` / conda, or install uv.")

def install(*args):
    if _has_pip():
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet",
                               "--disable-pip-version-check", *args])
    else:
        subprocess.check_call(["uv", "pip", "install", "--python", sys.executable,
                               "--quiet", *args])

install("--find-links", str(WHEELS), "arc-agi", "arcengine")
install("requests", "matplotlib", "numpy", "scipy", "imageio", "imageio-ffmpeg",
        "python-dotenv", "pillow")

# make the two source trees importable (avoids the repos' exact ==3.12.12 pin)
for p in (str(TAAF_SRC), str(INF_SRC)):
    if p not in sys.path:
        sys.path.insert(0, p)

import arc_agi, arcengine  # noqa: F401
print("deps OK: arc_agi", getattr(arc_agi, "__version__", "?"),
      "| arcengine", getattr(arcengine, "__version__", "?"))


## 3 · Wire the backend into the Duck agent

Environment variables must be set **before** the agent module is imported (it reads them at import time). This cell also fires a 1-token probe so key/model problems fail fast, with a hint per HTTP status.

In [ ]:
# ==============================================================================
#  Wire the chosen backend into the Duck agent (env vars are read at import
#  time by the agent module, so this cell MUST run before the next imports)
# ==============================================================================
import os, getpass, json, requests

api_key = ""
if B["api_key_env"]:
    api_key = os.environ.get(B["api_key_env"], "").strip()
    if not api_key:
        api_key = getpass.getpass(f"{B['api_key_env']} (input hidden, never stored): ").strip()
        os.environ[B["api_key_env"]] = api_key

os.environ.update({
    # --- what the Duck agent reads ------------------------------------------
    "LOCAL_ANALYZER_BASE_URL": B["base_url"],
    "OPENAI_BASE_URL": B["base_url"],
    "LOCAL_ANALYZER_PROVIDER": B["provider"],
    "OPENAI_PROVIDER": B["provider"],
    "LOCAL_ANALYZER_MODEL_ID": B["model"],
    "INFERENCE_ANALYZER_MODEL": B["model"],
    "LOCAL_ANALYZER_API_KEY": api_key,
    "LOCAL_ANALYZER_APP_NAME": "Duck Local Runner",
    "LOCAL_ANALYZER_CONTEXT_WINDOW": "32768",   # same working context as Kaggle
    "LOCAL_ANALYZER_MAX_OUTPUT": "0",
    "LOCAL_ANALYZER_TOOL_STEPS": "0",
    "LOCAL_ANALYZER_TOOL_TIMEOUT": "30",
    "LOCAL_ANALYZER_TOOL_OUTPUT_TOKENS": "1024",
    "LOCAL_ANALYZER_YIELD_SECONDS": "0",
    "LOCAL_ANALYZER_TEMPERATURE": "0.6",        # experiments may override later
    "LOCAL_ANALYZER_TOP_P": "0.95",
    "LOCAL_ANALYZER_TOP_K": "20",
    "LOCAL_ANALYZER_ENABLE_THINKING": "1",
    "MULTIMODAL_CONTEXT": "current_grid" if B["multimodal"] else "",
    "MULTIMODAL_UPSCALE": "4",
    # --- framework flags -------------------------------------------------------
    "MPLBACKEND": "Agg",
    "ONLY_RESET_LEVELS": "true",       # RESET restarts the level, not the game
    "TAAF_RUN_AS_SUBMISSION": "0",
    "TAAF_MINIMAL_DIAGNOSTICS": "0",
})

# --- fail-fast connectivity probe: one tiny completion --------------------------
try:
    r = requests.post(
        B["base_url"].rstrip("/") + "/chat/completions",
        headers={"Content-Type": "application/json",
                 **({"Authorization": f"Bearer {api_key}"} if api_key else {})},
        json={"model": B["model"], "max_tokens": 8,
              "messages": [{"role": "user", "content": "Say OK"}]},
        timeout=60,
    )
    if r.status_code == 200:
        msg = r.json()["choices"][0]["message"].get("content", "")
        print(f"backend probe OK -> {msg!r}")
    else:
        print(f"BACKEND PROBE FAILED: HTTP {r.status_code}: {r.text[:300]}\n"
              f"  401 -> bad/missing API key | 404 -> wrong model slug | "
              f"400 -> try provider='openrouter' in SETTINGS")
except Exception as exc:
    print(f"BACKEND PROBE FAILED: {exc!r}\n  (local server not running? wrong port?)")


## 4 · Choose the experiment

Same preset names as the Kaggle notebook; restart timings auto-scale from Kaggle's 132-minute games to your `MINUTES_PER_GAME`.

In [ ]:
# ==============================================================================
#  EXPERIMENTS - same names and semantics as duck-mods-v2-experiments.ipynb
# ==============================================================================
#  Restart timings were designed for Kaggle's 132-min game budget; here they are
#  auto-scaled to your MINUTES_PER_GAME so the *shape* of each experiment is
#  preserved (e.g. mods_v1 = ~4-5 attempt windows per game).
#  Kaggle-scheduling presets (long_waves / two_waves) don't exist here - locally
#  you control scheduling directly via MINUTES_PER_GAME and CONCURRENT_GAMES.

DEFAULTS = {
    "description": "Stock Duck behavior.",
    "stall_restart": {"enabled": False, "stall_minutes": 27.0,
                      "min_session_minutes": 6.0, "max_restarts": 4,
                      "keep_world_model": False},
    "prompt_addendum": False,
    "sampling": {"temperature": None},     # None = stock 0.6
}

EXPERIMENTS = {
    "stock":        {"description": "Control - identical to Tufa's agent."},
    "prompt_only":  {"description": "Mod 2 only.", "prompt_addendum": True},
    "restart_only": {"description": "Mod 1 only.", "stall_restart": {"enabled": True}},
    "mods_v1":      {"description": "RECOMMENDED: stall restarts + prompt addendum.",
                     "stall_restart": {"enabled": True}, "prompt_addendum": True},
    "restart_fast": {"description": "More, shallower attempts (18min/x6 at Kaggle scale).",
                     "stall_restart": {"enabled": True, "stall_minutes": 18.0,
                                       "min_session_minutes": 5.0, "max_restarts": 6},
                     "prompt_addendum": True},
    "restart_slow": {"description": "Fewer, deeper attempts (36min/x3 at Kaggle scale).",
                     "stall_restart": {"enabled": True, "stall_minutes": 36.0,
                                       "min_session_minutes": 8.0, "max_restarts": 3},
                     "prompt_addendum": True},
    "restart_max":  {"description": "Stress test (12min/x8 at Kaggle scale).",
                     "stall_restart": {"enabled": True, "stall_minutes": 12.0,
                                       "min_session_minutes": 4.0, "max_restarts": 8},
                     "prompt_addendum": True},
    "keep_world_model": {"description": "Restarts keep the world-model note.",
                     "stall_restart": {"enabled": True, "keep_world_model": True},
                     "prompt_addendum": True},
    "hot_sampling": {"description": "mods_v1 + temperature 0.7.",
                     "stall_restart": {"enabled": True}, "prompt_addendum": True,
                     "sampling": {"temperature": 0.7}},
    "cold_sampling": {"description": "mods_v1 + temperature 0.45.",
                     "stall_restart": {"enabled": True}, "prompt_addendum": True,
                     "sampling": {"temperature": 0.45}},
    "custom":       {"description": "Scratch slot - edit freely.",
                     "stall_restart": {"enabled": True}, "prompt_addendum": True},
}


def _resolve_config(name):
    if name not in EXPERIMENTS:
        raise KeyError(f"Unknown EXPERIMENT {name!r}. Choose one of: {sorted(EXPERIMENTS)}")
    import copy as _copy
    cfg = _copy.deepcopy(DEFAULTS)
    for key, value in EXPERIMENTS[name].items():
        if isinstance(value, dict) and isinstance(cfg.get(key), dict):
            cfg[key].update(value)
        else:
            cfg[key] = value
    cfg["experiment"] = name

    # scale Kaggle-calibrated restart timings to the local per-game budget
    sr = cfg["stall_restart"]
    scale = MINUTES_PER_GAME / 132.0
    sr["stall_minutes"] = max(3.0, round(sr["stall_minutes"] * scale, 1))
    sr["min_session_minutes"] = max(1.5, round(sr["min_session_minutes"] * scale, 1))
    return cfg


CONFIG = _resolve_config(EXPERIMENT)
_sr = CONFIG["stall_restart"]
print(f"EXPERIMENT {EXPERIMENT}: {CONFIG['description']}")
print(f"  stall_restart: {'ON' if _sr['enabled'] else 'off'}"
      + (f" - wipe after {_sr['stall_minutes']:g} min (scaled for a "
         f"{MINUTES_PER_GAME}-min game), warmup {_sr['min_session_minutes']:g} min, "
         f"max {_sr['max_restarts']}, keep_wm={_sr['keep_world_model']}" if _sr["enabled"] else ""))
print(f"  prompt_addendum: {CONFIG['prompt_addendum']} | temperature: "
      f"{CONFIG['sampling']['temperature'] or 'stock 0.6'}")


## 5 · Build the benchmark (offline games + solver)

In [ ]:
# ==============================================================================
#  Build the benchmark: offline games + solver + output directory
# ==============================================================================
import time
import arc_agi
import taaf.game_api, taaf.benchmark, taaf.deploy_inline
from inference.framework.solver import HarnessSolver

spec = taaf.game_api.ArcadeSpec(operation_mode=arc_agi.OperationMode.OFFLINE,
                                environments_dir=str(ENV_DIR))
arcade = arc_agi.Arcade(operation_mode=arc_agi.OperationMode.OFFLINE,
                        environments_dir=str(ENV_DIR))
all_ids = sorted(e.game_id for e in arcade.available_environments)

if GAMES == "all":
    game_ids = all_ids
else:
    game_ids = [gid for gid in all_ids if any(gid.startswith(p) for p in GAMES)]
    missing = [p for p in GAMES if not any(g.startswith(p) for g in all_ids)]
    if missing:
        print(f"WARNING: no game matches prefixes {missing}")
if not game_ids:
    raise SystemExit("No games selected.")

games = [taaf.game_api.GameAPI(env_name=g, arcade_spec=spec) for g in game_ids]

solver = HarnessSolver(
    label=f"duck-local-{EXPERIMENT}",
    model=B["model"],
    concurrency=int(CONCURRENT_GAMES),
    max_runtime_s_per_game=float(MINUTES_PER_GAME) * 60.0,
    analyzer_timeout=300.0,
    save_request_logs=False,
)

bm = taaf.benchmark.Benchmark(games=games, solver=solver,
                              label=f"local-{EXPERIMENT}", n_passes=int(N_PASSES))
RUN_DIR = OUTPUT_ROOT / time.strftime(f"%Y%m%d_%H%M%S_{EXPERIMENT}")
RUN_DIR.mkdir(parents=True, exist_ok=True)
bm.job_dir = RUN_DIR

target = taaf.deploy_inline.InlineTarget()
target.max_runtime_s = float(TOTAL_HOURS_CAP) * 3600.0 if TOTAL_HOURS_CAP else 10 * 24 * 3600.0

print(f"{len(games)} game(s) x {N_PASSES} pass(es) = {len(games)*N_PASSES} runs | "
      f"{MINUTES_PER_GAME} min/game | {CONCURRENT_GAMES} concurrent")
print("run dir:", RUN_DIR.resolve())


## 6 · Install the DUCK MODS engine

Same defensively-monkeypatched engine as the Kaggle notebook: stall restarts (`[duck-mods] … restart #n` lines will appear live), prompt addendum, temperature. `DUCK_STATS` collects per-game telemetry.

In [ ]:
# ==============================================================================
#  DUCK MODS ENGINE (same tested engine as the Kaggle notebook)
# ==============================================================================
DUCK_STATS = {"experiment": CONFIG["experiment"], "games": {}, "applied": [], "failed": []}


def _install_duck_mods(cfg, stats):
    import time as _time
    import traceback as _tb

    applied, failed = stats["applied"], stats["failed"]
    try:
        import inference.agent.tool_agent as _ta
    except Exception:
        print("[duck-mods] cannot import tool_agent - no mods applied:\n" + _tb.format_exc(), flush=True)
        return

    # ---- prompt addendum -------------------------------------------------------
    if cfg.get("prompt_addendum"):
        try:
            _addendum = (
                "\n\nStall recovery and scoring mechanics:\n"
                "- The per-level score is (human_actions / your_actions)^2, so wasted actions are punished "
                "quadratically. Completing a new level is worth far more than polishing an old one, and "
                "levels already completed stay completed even after RESET.\n"
                "- If many turns pass with no level completion, treat your current theory of the game as "
                "probably wrong: write down 2-3 alternative interpretations of the mechanics and run the "
                "cheapest discriminating probe, instead of repeating variations of the same failing plan.\n"
                "- If the level looks unwinnable from the current position, or you have already spent a very "
                "large number of actions on it, RESET the level and execute your best-known route efficiently.\n"
            )
            if hasattr(_ta, "PYTHON_ADDENDUM") and _addendum not in _ta.PYTHON_ADDENDUM:
                _ta.PYTHON_ADDENDUM = _ta.PYTHON_ADDENDUM + _addendum
                applied.append("prompt_addendum")
        except Exception:
            failed.append("prompt_addendum"); print(_tb.format_exc(), flush=True)

    # ---- temperature -------------------------------------------------------------
    try:
        _temp = cfg.get("sampling", {}).get("temperature")
        if _temp is not None:
            _ta._LOCAL_ANALYZER_TEMPERATURE = float(_temp)
            applied.append("temperature=%g" % _temp)
    except Exception:
        failed.append("temperature"); print(_tb.format_exc(), flush=True)

    # ---- stall restart --------------------------------------------------------------
    _sr = cfg.get("stall_restart", {})
    if _sr.get("enabled"):
        try:
            if getattr(_ta.ToolAgent, "_duck_mods_patched", False):
                applied.append("stall_restart(already patched)")
            else:
                _orig_analyze = _ta.ToolAgent.analyze
                _stall_s = float(_sr["stall_minutes"]) * 60.0
                _warmup_s = float(_sr["min_session_minutes"]) * 60.0
                _max_restarts = int(_sr["max_restarts"])
                _keep_wm = bool(_sr["keep_world_model"])
                _games = stats["games"]

                def _analyze_with_restarts(self, state_path, action_num, **kwargs):
                    key = None
                    try:
                        key = str(state_path.parent)
                        now = _time.monotonic()
                        tr = _games.get(key)
                        if tr is None:
                            tr = _games[key] = {"name": state_path.parent.name,
                                                "t0": now, "level": None,
                                                "t_prog": now, "restarts": 0}
                        if (tr["restarts"] < _max_restarts
                                and (now - tr["t0"]) >= _warmup_s
                                and (now - tr["t_prog"]) >= _stall_s
                                and self._history_messages):
                            self._history_messages = []
                            self._last_step_summary = None
                            if not _keep_wm:
                                try:
                                    self._summarized_knowledge = _ta._empty_world_model()
                                except Exception:
                                    pass
                            tr["restarts"] += 1
                            tr["t_prog"] = now
                            print("[duck-mods] %s: fresh-context restart #%d after %.1f min "
                                  "without level progress"
                                  % (tr["name"], tr["restarts"], _stall_s / 60.0), flush=True)
                    except Exception:
                        print("[duck-mods] restart check failed:\n" + _tb.format_exc(), flush=True)

                    result = _orig_analyze(self, state_path, action_num, **kwargs)

                    try:
                        lar = getattr(self, "_last_action_result", None) or {}
                        lvl = lar.get("level")
                        tr = _games.get(key) if key is not None else None
                        if tr is not None and isinstance(lvl, int):
                            if tr["level"] is None or lvl > tr["level"]:
                                tr["level"] = lvl
                                tr["t_prog"] = _time.monotonic()
                    except Exception:
                        pass
                    return result

                _ta.ToolAgent.analyze = _analyze_with_restarts
                _ta.ToolAgent._duck_mods_patched = True
                applied.append("stall_restart(%gmin, warmup %gmin, max %d, keep_wm=%s)"
                               % (_sr["stall_minutes"], _sr["min_session_minutes"],
                                  _max_restarts, _keep_wm))
        except Exception:
            failed.append("stall_restart"); print(_tb.format_exc(), flush=True)

    print("[duck-mods] applied: %s | failed: %s" % (applied or "none", failed or "none"), flush=True)


_install_duck_mods(CONFIG, DUCK_STATS)


## 7 · Run

In [ ]:
# ==============================================================================
#  RUN  (Jupyter supports top-level await; this can take a while - watch the
#  [duck-mods] restart lines and per-game progress in the output)
# ==============================================================================
from datetime import datetime, timedelta

soft_end = None
if TOTAL_HOURS_CAP:
    soft_end = datetime.now() + timedelta(hours=float(TOTAL_HOURS_CAP))

await bm.run(soft_end_time=soft_end, runtime_environment=target, minimal_diagnostics=False)
print("\nbenchmark finished:", bm.label)


## 8 · Results — official framework scores + telemetry + chart

In [ ]:
# ==============================================================================
#  RESULTS - official framework scores (same formula as the Kaggle scorecard:
#  per level min(115, (human/agent)^2 x 100), level-weighted, completion-capped)
# ==============================================================================
import json, statistics
import matplotlib.pyplot as plt

per_game = {}
for run in bm.game_runs:
    score = run.final_score
    if score is None:
        try:
            score = run._compute_final_score()
        except Exception:
            score = 0.0
    per_game.setdefault(run.game_id, []).append(
        {"score": float(score or 0.0), "levels": int(run.levels_completed),
         "state": str(run.state)})

rows = []
for gid in sorted(per_game):
    runs = per_game[gid]
    scores = [r["score"] for r in runs]
    rows.append({"game": gid, "passes": len(runs),
                 "mean": statistics.mean(scores), "best": max(scores),
                 "levels_best": max(r["levels"] for r in runs)})

overall = statistics.mean(r["mean"] for r in rows) if rows else 0.0
print(f"=== {EXPERIMENT} | backend={BACKEND} | model={B['model']} ===")
print(f"OVERALL (mean of per-game means): {overall:.3f}")
print(f"{'game':<16}{'passes':>7}{'mean':>9}{'best':>9}{'levels':>8}")
for r in rows:
    print(f"{r['game']:<16}{r['passes']:>7}{r['mean']:>9.3f}{r['best']:>9.3f}{r['levels_best']:>8}")

# restart telemetry
games = DUCK_STATS["games"]
total_restarts = sum(t["restarts"] for t in games.values())
print(f"\n[duck-mods] restarts fired: {total_restarts} across {len(games)} tracked game-runs")

# persist a machine-readable record for your experiment log
record = {"experiment": EXPERIMENT, "backend": BACKEND, "model": B["model"],
          "n_passes": N_PASSES, "minutes_per_game": MINUTES_PER_GAME,
          "concurrent": CONCURRENT_GAMES, "overall": overall, "games": rows,
          "restarts": total_restarts, "config": CONFIG}
(RUN_DIR / "results.json").write_text(json.dumps(record, indent=2, default=str))
print("saved:", (RUN_DIR / "results.json").resolve())

# quick visual
if rows:
    fig, ax = plt.subplots(figsize=(10, max(3, 0.32 * len(rows))))
    names = [r["game"][:4] for r in rows]
    ax.barh(names, [r["mean"] for r in rows], color="#5eead4", label="mean")
    ax.barh(names, [r["best"] for r in rows], color="none", edgecolor="#818cf8",
            linewidth=1.4, label="best pass")
    ax.set_xlabel("ARC score"); ax.set_title(f"{EXPERIMENT} - per-game scores")
    ax.legend(); ax.invert_yaxis(); plt.tight_layout(); plt.show()


## 9 · Optional: full diagnostics viewer

In [ ]:
# ==============================================================================
#  Optional: rich per-run diagnostics written by the framework
# ==============================================================================
from html import escape
from IPython.display import HTML, display

diag = RUN_DIR / "diagnostics.html"
if diag.is_file():
    display(HTML(f'<iframe srcdoc="{escape(diag.read_text(), quote=True)}" '
                 'width="100%" height="850" style="border:0"></iframe>'))
else:
    print("no diagnostics.html in", RUN_DIR)


## 10 · A/B protocol & troubleshooting

**Protocol.** Fix backend/model/games/minutes. Run `stock` and `mods_v1` with `N_PASSES ≥ 5` each.
Promote a config to Kaggle only if its overall mean beats the incumbent by more than one
between-pass standard deviation (public-set σ per pass ≈ 0.45 at Kaggle scale — variance is brutal,
do not trust single runs). Then sweep one knob at a time: `restart_fast` vs `restart_slow` vs
`restart_max`, then `hot_sampling` vs `cold_sampling`, then `keep_world_model`.

**Cheap iteration recipe:** `GAMES = ["ft09","vc33","r11l","sb26","lp85"]` (the games where the Duck
actually scores) with `N_PASSES = 5`, `MINUTES_PER_GAME = 15` — a full A/B in an evening on OpenRouter.

**Troubleshooting**

| Symptom | Fix |
|---|---|
| HTTP 401 on probe | wrong/missing API key (check the `api_key_env` variable) |
| HTTP 404 on probe | wrong model slug — check the provider's model catalog |
| HTTP 400 on probe / during run | server rejects extra sampling fields → set `provider": "openrouter"` in SETTINGS |
| local model replies but plays terribly | it's text-only → `multimodal: False`, or too small — the Duck was tuned on 27B-class VLMs |
| rate-limit errors | lower `CONCURRENT_GAMES` to 2, or add credits |
| `SyntaxError` in taaf/inference imports | kernel is not Python 3.12 |
| run seems stuck | it's thinking — watch `local-runs/<run>/…` artifacts and the printed action lines |

**Mapping to Kaggle:** the winner preset goes into `duck-mods-v2-experiments.ipynb` (same `EXPERIMENT`
name) → smoke-test on Kaggle (Save & Run) → submit daily. Full strategy: `index.html`, tabs 4–5.